# Merging, Aligning, and Cleaning Financial Time Series Data

In this lab we combine two different financial time series datasets into one clean analysis-ready dataset:

- `equity_fixed_income.csv`: daily end-of-day traditional market data
- `crypto.csv`: hourly cryptocurrency price data

The workflow covers loading, inspection, missing values, timezone handling, daily resampling, calendar alignment, normalization, visualization, and a light risk/reward analysis.

If the CSV files are not available, the notebook creates reproducible synthetic data so the lab still runs from top to bottom.

## 1) Import Libraries

We use only `pandas`, `numpy`, and `matplotlib` for this lab.

In [5]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:,.4f}'.format
plt.style.use('default')

## 2) Helper Functions

Real-world CSV files do not always use the same datetime column name. These small helper functions keep the notebook robust without hiding the main workflow.

In [6]:
def find_csv_file(filename):
    """Find a CSV either in the current working directory or nearby project folders."""
    candidate_paths = [
        Path(filename),
        Path('pandas-financial-data-analysis') / filename,
        Path.cwd().parent / filename,
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    return None


def generate_synthetic_equity_fixed_income_data():
    """Create reproducible daily equity/fixed-income style data when the CSV is unavailable."""
    rng = np.random.default_rng(42)
    dates = pd.bdate_range(start='2022-01-03', end='2024-12-31')

    synthetic_specs = {
        'US_Equity': {'start': 100, 'daily_mean': 0.00035, 'daily_vol': 0.0110},
        'Europe_Equity': {'start': 100, 'daily_mean': 0.00025, 'daily_vol': 0.0100},
        'Emerging_Markets': {'start': 100, 'daily_mean': 0.00020, 'daily_vol': 0.0140},
        'Government_Bonds': {'start': 100, 'daily_mean': 0.00008, 'daily_vol': 0.0030},
        'Corporate_Bonds': {'start': 100, 'daily_mean': 0.00012, 'daily_vol': 0.0045},
    }

    data = pd.DataFrame(index=dates)

    for column, spec in synthetic_specs.items():
        returns = rng.normal(spec['daily_mean'], spec['daily_vol'], len(dates))
        data[column] = spec['start'] * np.cumprod(1 + returns)

    data.index.name = 'Timestamp'
    return data


def generate_synthetic_crypto_data():
    """Create reproducible hourly crypto style data when the CSV is unavailable."""
    rng = np.random.default_rng(123)
    timestamps = pd.date_range(start='2022-01-03 00:00', end='2024-12-31 23:00', freq='h')

    synthetic_specs = {
        'BTC': {'start': 47000, 'hourly_mean': 0.00004, 'hourly_vol': 0.0080},
        'ETH': {'start': 3800, 'hourly_mean': 0.00005, 'hourly_vol': 0.0100},
    }

    data = pd.DataFrame(index=timestamps)

    for column, spec in synthetic_specs.items():
        returns = rng.normal(spec['hourly_mean'], spec['hourly_vol'], len(timestamps))
        data[column] = spec['start'] * np.cumprod(1 + returns)

    data.index.name = 'Timestamp'
    return data


def generate_synthetic_data(filename):
    """Choose the right synthetic fallback based on the requested CSV name."""
    if filename == 'equity_fixed_income.csv':
        return generate_synthetic_equity_fixed_income_data(), 'synthetic equity/fixed income data', 'Timestamp'
    if filename == 'crypto.csv':
        return generate_synthetic_crypto_data(), 'synthetic hourly crypto data', 'Timestamp'

    raise FileNotFoundError(f'No CSV file or synthetic fallback is configured for {filename}.')


def find_datetime_column(dataframe):
    """Try common datetime column names, then fall back to the first parsable column."""
    common_names = ['Date', 'Datetime', 'DateTime', 'Timestamp', 'timestamp', 'time', 'Time']

    for column in common_names:
        if column in dataframe.columns:
            return column

    for column in dataframe.columns:
        if pd.api.types.is_numeric_dtype(dataframe[column]):
            continue

        parsed = pd.to_datetime(dataframe[column], errors='coerce')
        if parsed.notna().mean() > 0.80:
            return column

    raise ValueError('Could not identify a datetime column. Inspect the CSV columns manually.')


def load_time_series_csv(filename):
    """Load a CSV, parse its datetime column, set it as index, and sort chronologically."""
    path = find_csv_file(filename)

    if path is None:
        synthetic_data, source_label, datetime_column = generate_synthetic_data(filename)
        print(f'Could not find {filename}. Using {source_label} instead.')
        return synthetic_data, source_label, datetime_column

    raw_data = pd.read_csv(path)
    datetime_column = find_datetime_column(raw_data)

    raw_data[datetime_column] = pd.to_datetime(raw_data[datetime_column], errors='coerce')
    cleaned_data = raw_data.dropna(subset=[datetime_column]).set_index(datetime_column).sort_index()
    cleaned_data.index.name = 'Timestamp'

    return cleaned_data, path, datetime_column


def numeric_price_columns(dataframe):
    """Return numeric columns only, coercing object columns where possible."""
    numeric_data = dataframe.copy()

    for column in numeric_data.columns:
        numeric_data[column] = pd.to_numeric(numeric_data[column], errors='coerce')

    numeric_data = numeric_data.select_dtypes(include='number')

    if numeric_data.empty:
        raise ValueError('No numeric price columns were found.')

    return numeric_data


def describe_index_frequency(index):
    """Give a simple daily/hourly style description based on median spacing."""
    if len(index) < 2:
        return 'Not enough rows to infer frequency'

    median_gap = index.to_series().diff().dropna().median()

    if median_gap <= pd.Timedelta(hours=2):
        return f'Appears intraday or hourly-like, median gap = {median_gap}'
    if median_gap <= pd.Timedelta(days=2):
        return f'Appears daily-like, median gap = {median_gap}'

    return f'Appears lower frequency, median gap = {median_gap}'

## 3) Load and Inspect Both Datasets

The first task in any data preparation workflow is inspection. We need to confirm shape, columns, index type, and whether the data looks daily or hourly.

This notebook first looks for the requested CSV files. If they are missing, it uses dummy financial data with realistic daily and hourly frequencies.

In [7]:
equity_data, equity_path, equity_datetime_column = load_time_series_csv('equity_fixed_income.csv')
crypto_data, crypto_path, crypto_datetime_column = load_time_series_csv('crypto.csv')

print(f'Equity/fixed income file: {equity_path}')
print(f'Equity datetime column used: {equity_datetime_column}')
print(f'Crypto file: {crypto_path}')
print(f'Crypto datetime column used: {crypto_datetime_column}')

Could not find equity_fixed_income.csv. Using synthetic equity/fixed income data instead.
Could not find crypto.csv. Using synthetic hourly crypto data instead.
Equity/fixed income file: synthetic equity/fixed income data
Equity datetime column used: Timestamp
Crypto file: synthetic hourly crypto data
Crypto datetime column used: Timestamp


In [8]:
print('Equity/fixed income head:')
print(equity_data.head())

print('\nEquity/fixed income info:')
equity_data.info()

print('\nEquity/fixed income shape:', equity_data.shape)
print('Equity index type:', type(equity_data.index))
print(describe_index_frequency(equity_data.index))

Equity/fixed income head:
            US_Equity  Europe_Equity  Emerging_Markets  Government_Bonds  \
Timestamp                                                                  
2022-01-03   100.3702       101.8226           98.7037           99.9551   
2022-01-04    99.2571       102.7585           97.8462           99.7556   
2022-01-05   100.1112       102.7960           96.2720          100.2131   
2022-01-06   101.1820       103.0774           96.5097          100.4465   
2022-01-07    99.0459       103.1488           97.5149          100.6508   

            Corporate_Bonds  
Timestamp                    
2022-01-03          99.6852  
2022-01-04          99.8630  
2022-01-05         100.1754  
2022-01-06          99.8855  
2022-01-07          99.9208  

Equity/fixed income info:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 782 entries, 2022-01-03 to 2024-12-31
Freq: B
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            ---

In [9]:
print('Crypto head:')
print(crypto_data.head())

print('\nCrypto info:')
crypto_data.info()

print('\nCrypto shape:', crypto_data.shape)
print('Crypto index type:', type(crypto_data.index))
print(describe_index_frequency(crypto_data.index))

Crypto head:
                            BTC        ETH
Timestamp                                 
2022-01-03 00:00:00 46,629.9704 3,764.6975
2022-01-03 01:00:00 46,494.6365 3,772.6919
2022-01-03 02:00:00 46,975.5492 3,814.3993
2022-01-03 03:00:00 47,050.3247 3,750.0145
2022-01-03 04:00:00 47,398.5840 3,796.0799

Crypto info:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26256 entries, 2022-01-03 00:00:00 to 2024-12-31 23:00:00
Freq: h
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   BTC     26256 non-null  float64
 1   ETH     26256 non-null  float64
dtypes: float64(2)
memory usage: 615.4 KB

Crypto shape: (26256, 2)
Crypto index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Appears intraday or hourly-like, median gap = 0 days 01:00:00


## 4) Analyze and Handle Missing Values

Financial price datasets often contain missing values because of market holidays, asset-specific trading calendars, API gaps, or merged data from different sources.

For price series, forward fill is often sensible because it carries the last known observed price forward until a new price is available. This is not always appropriate for every dataset, but it is a common starting point for aligned price analysis.

In [ ]:
equity_prices = numeric_price_columns(equity_data)
crypto_prices = numeric_price_columns(crypto_data)

missing_summary = pd.DataFrame(
    {
        'equity_missing': equity_prices.isna().sum(),
        'crypto_missing': crypto_prices.isna().sum(),
    }
)

missing_summary

In [ ]:
# Forward fill price columns to carry the last known price forward.
equity_prices_clean = equity_prices.ffill()
crypto_prices_clean = crypto_prices.ffill()

print('Remaining missing values after forward fill:')
print('Equity/fixed income:', int(equity_prices_clean.isna().sum().sum()))
print('Crypto:', int(crypto_prices_clean.isna().sum().sum()))

## 5) Handle Crypto Time Zones

Before merging hourly crypto data with daily traditional market data, we need to make the crypto timestamps timezone-aware.

For this lab, if the crypto index is timezone-naive, we assume it represents `Europe/Berlin` time. We then convert it to `US/Eastern` so the data aligns with a US-style market calendar.

In [ ]:
print('Crypto index timezone before localization/conversion:')
print(crypto_prices_clean.index.tz)

print('\nCrypto rows before timezone handling:')
print(crypto_prices_clean.head())

In [ ]:
if crypto_prices_clean.index.tz is None:
    # Localize naive timestamps. DST transitions can create nonexistent or ambiguous hours,
    # so we handle those cases explicitly for robust hourly data processing.
    crypto_berlin = crypto_prices_clean.tz_localize(
        'Europe/Berlin',
        nonexistent='shift_forward',
        ambiguous='NaT',
    )
    crypto_berlin = crypto_berlin[~crypto_berlin.index.isna()]
    crypto_berlin = crypto_berlin.groupby(level=0).last()
else:
    crypto_berlin = crypto_prices_clean.copy()

# Convert timezone-aware crypto timestamps to US/Eastern for calendar alignment.
crypto_eastern = crypto_berlin.tz_convert('US/Eastern')

print('Crypto rows after conversion to US/Eastern:')
print(crypto_eastern.head())

print('\nCrypto index timezone after conversion:')
print(crypto_eastern.index.tz)

Timezone alignment matters because the same clock label can represent different absolute moments in different locations. If we merge before fixing time zones, observations may line up on the wrong day or wrong session.

## 6) Downsample Hourly Crypto Data to Daily End-of-Day Data

The equity/fixed income data is daily, while the crypto data is hourly. To merge them cleanly, we downsample crypto to daily frequency using the last available observation of each day.

For price data, the last observation is a reasonable proxy for an end-of-day close.

In [ ]:
crypto_daily = crypto_eastern.resample('D').last().ffill()

# Remove timezone and time component after daily resampling so we can align by date.
crypto_daily.index = crypto_daily.index.tz_localize(None).normalize()
crypto_daily.index.name = 'Date'

# Prefix crypto columns so they remain clearly identifiable after merging.
crypto_daily = crypto_daily.add_prefix('Crypto_')

crypto_daily.head()

## 7) Align Calendars and Merge Datasets

Traditional market assets do not trade every calendar day, while crypto trades continuously. We treat the equity/fixed income dataset as the reference calendar and align daily crypto data to those same dates.

In [ ]:
# Normalize the equity/fixed income index to date-only values.
equity_prices_clean = equity_prices_clean.copy()
equity_prices_clean.index = pd.to_datetime(equity_prices_clean.index).normalize()
equity_prices_clean.index.name = 'Date'

# Remove duplicate dates defensively by keeping the last observation per date.
equity_daily = equity_prices_clean.groupby(level=0).last().ffill()

reference_calendar = equity_daily.index
crypto_aligned = crypto_daily.reindex(reference_calendar).ffill()

merged_prices = pd.concat([equity_daily, crypto_aligned], axis=1)
merged_prices = merged_prices.ffill().dropna(how='all')

merged_prices.head()

In [ ]:
print('Merged tail:')
print(merged_prices.tail())

print('\nMerged info:')
merged_prices.info()

print('\nMissing values after merge:')
print(merged_prices.isna().sum())

In [ ]:
# Drop columns that still have no usable observations, then drop leading rows with missing prices.
merged_prices = merged_prices.dropna(axis=1, how='all').dropna(how='any')

print('Clean merged shape:', merged_prices.shape)
print('Remaining missing values:', int(merged_prices.isna().sum().sum()))
merged_prices.head()

## 8) Normalize the Price Series

Different assets have different price levels, so raw price charts can be misleading. Normalization makes all series start at `100`, allowing us to compare relative performance.

Conceptually:

`normalized = price / first_price * 100`

In [ ]:
first_valid_prices = merged_prices.apply(lambda column: column.dropna().iloc[0])
normalized_prices = merged_prices.divide(first_valid_prices).multiply(100)

normalized_prices.head()

## 9) Visualize Normalized Performance

The chart below compares all assets on the same starting base of `100`.

In [ ]:
plt.figure(figsize=(13, 7))

for column in normalized_prices.columns:
    plt.plot(normalized_prices.index, normalized_prices[column], label=column)

plt.title('Normalized Asset Performance')
plt.xlabel('Date')
plt.ylabel('Normalized Price, Base = 100')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

## 10) Light Exploratory Analysis

Finally, we compute simple percentage returns and use them for a first-pass risk/reward comparison.

Here we define:
- performance as total normalized growth over the sample
- volatility as the standard deviation of daily simple returns

In [ ]:
simple_returns = merged_prices.pct_change(fill_method=None).dropna(how='all')

total_return = merged_prices.iloc[-1] / merged_prices.iloc[0] - 1
daily_volatility = simple_returns.std()

analysis_summary = pd.DataFrame(
    {
        'Total Return': total_return,
        'Daily Volatility': daily_volatility,
    }
).sort_values(by='Total Return', ascending=False)

analysis_summary

In [ ]:
best_asset = total_return.idxmax()
best_asset_return = total_return.loc[best_asset]

most_volatile_asset = daily_volatility.idxmax()
most_volatile_value = daily_volatility.loc[most_volatile_asset]

print(f'Best performing asset: {best_asset} ({best_asset_return:.2%} total return)')
print(f'Most volatile asset: {most_volatile_asset} ({most_volatile_value:.2%} daily return std)')

## Final Conclusion

In this lab we loaded daily equity/fixed income data and hourly crypto data, inspected their structure, handled missing values, localized and converted crypto timestamps, resampled hourly crypto prices to daily end-of-day observations, aligned the crypto data to the traditional market calendar, and merged everything into one clean dataset.

Timezone conversion was necessary because hourly timestamps from different regions can represent different absolute moments. Resampling was necessary because one dataset was hourly and the other was daily. Calendar alignment was necessary because traditional markets and crypto markets do not share the same trading calendar.

The best-performing asset and most volatile asset are printed in the previous section based on the data used in the notebook. If the CSV files were unavailable, those results come from the synthetic fallback data.